# Gemini API playground

Test a Gemini API key and try drafting ticket replies before Cooper has built-in AI (FR-AI-003). Standard library only.

1. Create a key at <https://aistudio.google.com> → **Get API key**. Without a billing account the project stays on the **free tier**.
2. Put it in `.env` as `GEMINI_API_KEY=...` (never in this notebook, `.env` is git-ignored). Optionally set `GEMINI_MODEL`.
3. Run the cells top to bottom.

Free tier: limits are per Google Cloud project and shown in AI Studio (HTTP 429 when exceeded); Google may use free-tier prompts to improve its models, so **do not send real ticket data** on the free tier.

In [ ]:
import json
import os
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path


def load_env(name=".env"):
    """Load KEY=VALUE lines from the nearest .env upwards; real environment variables win."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        path = directory / name
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, value = line.partition("=")
                os.environ.setdefault(key.strip(), value.strip().strip("\"'"))
            return path
    return None

print("Loaded", load_env() or "no .env found (using the process environment)")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
# Pin a version: the *-latest aliases jump to the newest release, which is often overloaded.
GEMINI_MODEL = os.getenv("GEMINI_MODEL") or "gemini-3.6-flash"
# Tried in order when GEMINI_MODEL is overloaded (HTTP 503/429) or does not answer.
GEMINI_FALLBACK_MODELS = [m.strip() for m in (os.getenv("GEMINI_FALLBACK_MODELS") or "gemini-3-flash-preview,gemini-flash-lite-latest").split(",") if m.strip()]
GEMINI_BASE = "https://generativelanguage.googleapis.com/v1beta"

# Optional: a Cooper API Key (Administration -> API Keys) for the last section.
COOPER_URL = (os.getenv("COOPER_PUBLIC_URL") or "http://localhost:8080").rstrip("/")
COOPER_API_KEY = os.getenv("COOPER_API_KEY", "")


def masked(secret):
    return f"…{secret[-4:]} ({len(secret)} chars)" if len(secret) > 8 else "(not set)"

print("GEMINI_API_KEY:", masked(GEMINI_API_KEY))
print("GEMINI_MODEL:  ", GEMINI_MODEL, "| fallbacks:", ", ".join(GEMINI_FALLBACK_MODELS) or "none")
print("COOPER_URL:    ", COOPER_URL, "| COOPER_API_KEY:", masked(COOPER_API_KEY))
assert GEMINI_API_KEY, "Set GEMINI_API_KEY in .env, then re-run this cell (restart the kernel if you changed .env)."

In [ ]:
import time

RETRYABLE = {429, 500, 502, 503, 504}


class GeminiUnavailable(RuntimeError):
    """The model is overloaded, rate limited or too slow; another model may still answer."""


def http_json(url, headers=None, body=None, timeout=30, retries=2):
    """GET (or POST when body is given) JSON. Retries 429/5xx with backoff.

    A timeout is not retried: an overloaded model queues the request and never answers,
    so waiting again only wastes time. The caller can try another model instead.
    """
    data = json.dumps(body).encode() if body is not None else None
    short_url = url.split("?")[0]
    for attempt in range(retries + 1):
        request = urllib.request.Request(
            url, data=data, method="POST" if data is not None else "GET",
            headers={"Content-Type": "application/json", **(headers or {})},
        )
        try:
            with urllib.request.urlopen(request, timeout=timeout) as response:
                return json.loads(response.read() or b"null")
        except urllib.error.HTTPError as error:
            detail = error.read().decode(errors="replace")
            if error.code not in RETRYABLE:
                raise RuntimeError(f"HTTP {error.code} from {short_url}: {detail}") from None
            problem, wait = f"HTTP {error.code}: {detail.strip()[:300]}", float(error.headers.get("Retry-After") or 2 ** (attempt + 1))
        except TimeoutError:
            raise GeminiUnavailable(f"{short_url}: no response within {timeout}s (model overloaded)") from None
        except urllib.error.URLError as error:
            problem, wait = f"network error: {error.reason}", 2 ** (attempt + 1)
        if attempt < retries:
            print(f"  {problem} — retrying in {wait:.0f}s ({attempt + 1}/{retries})")
            time.sleep(wait)
    raise GeminiUnavailable(f"{short_url}: {problem}")


def gemini(path, body=None, **kwargs):
    return http_json(f"{GEMINI_BASE}/{path}", {"x-goog-api-key": GEMINI_API_KEY}, body, **kwargs)

## 1. Is the key valid? List the models it can use

An invalid key gives HTTP 400 `API_KEY_INVALID`; a key without the Generative Language API enabled gives 403.

In [ ]:
models, token = [], None
while True:
    query = urllib.parse.urlencode({"pageSize": 100, **({"pageToken": token} if token else {})})
    page = gemini(f"models?{query}")
    models += page.get("models", [])
    token = page.get("nextPageToken")
    if not token:
        break

text_models = [m for m in models if "generateContent" in m.get("supportedGenerationMethods", [])]
print(f"Key OK: {len(text_models)} models support generateContent\n")
for m in text_models:
    print(f"{m['name'].removeprefix('models/'):45} {m.get('inputTokenLimit', '?'):>9} in  {m.get('outputTokenLimit', '?'):>7} out")

## 2. Generate text

`generate` tries `GEMINI_MODEL` first and moves on to `GEMINI_FALLBACK_MODELS` when a model is overloaded (503), rate limited (429) or does not answer within the timeout.

Use a **pinned** model such as `gemini-3.6-flash`. The `gemini-flash-latest` alias points to the newest Flash release, and right after a release Google's capacity for it can be so short that requests hang with no response at all (no headers, even when streaming) or come back as 503 "high demand". Older 2.5 models return 404 for new keys. If a model gives 404, pick another from the list above.

In [ ]:
def generate(prompt, model=None, system=None, temperature=0.3):
    body = {
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": temperature},
    }
    if system:
        body["systemInstruction"] = {"parts": [{"text": system}]}
    candidates = [model] if model else [GEMINI_MODEL, *GEMINI_FALLBACK_MODELS]
    for index, name in enumerate(candidates):
        try:
            result = gemini(f"models/{name}:generateContent", body)
        except GeminiUnavailable as error:
            print(f"{name} unavailable ({error})")
            if index + 1 < len(candidates):
                continue
            raise
        candidate = (result.get("candidates") or [{}])[0]
        text = "".join(p.get("text", "") for p in candidate.get("content", {}).get("parts", []))
        print(f"[answered by {result.get('modelVersion', name)}]")
        return text.strip(), candidate.get("finishReason"), result.get("usageMetadata", {})

text, finish, usage = generate("Reply with one short sentence: is this Gemini API key working?")
print(text)
print("\nfinish:", finish, "| tokens:", usage.get("promptTokenCount"), "in,", usage.get("candidatesTokenCount"), "out")

## 3. Draft a reply to a sample ticket

Made-up ticket, so nothing real leaves your machine. Tweak `SYSTEM` to try prompts for FR-AI-003.

In [ ]:
SYSTEM = (
    "You are an Analyst on an IT service desk. Write a short, polite first reply to the Requester. "
    "Ask for any missing details. Never invent facts, never promise dates, and do not include a signature."
)

sample_ticket = {
    "number": 1042,
    "subject": "Outlook keeps asking for my password",
    "description": "Since this morning Outlook on my laptop asks for my password every few minutes. "
                   "Webmail works fine. I'm on Windows 11.",
}

draft, finish, usage = generate(
    f"Ticket {sample_ticket['number']}\nSubject: {sample_ticket['subject']}\n\n{sample_ticket['description']}",
    system=SYSTEM,
)
print(draft)
print("\nfinish:", finish, "| tokens:", usage.get("totalTokenCount"))

## 4. Optional: draft for the newest open ticket in local Cooper

Needs `cooper-server` running and `COOPER_API_KEY` in `.env` (an API Key that can read tickets and, to post, add Notes). Confidential tickets are skipped. By default the draft is only printed; set `POST_AS_NOTE = True` to add it as an **internal** Note. On the free tier, use test data only.

In [ ]:
POST_AS_NOTE = False


def cooper(path, body=None):
    return http_json(f"{COOPER_URL}/api/v1{path}", {"Authorization": f"Bearer {COOPER_API_KEY}"}, body)


if not COOPER_API_KEY:
    print("COOPER_API_KEY is not set: skipping.")
else:
    tickets = cooper("/tickets?open=true&sort=created_at&order=desc&limit=20")["items"]
    ticket = next((t for t in tickets if t.get("sensitivity") != "confidential"), None)
    if ticket is None:
        print("No open, non-Confidential tickets.")
    else:
        draft, finish, _ = generate(
            f"Ticket {ticket['number']}\nSubject: {ticket['subject']}\n\n{ticket.get('description') or ''}",
            system=SYSTEM,
        )
        print(f"Ticket {ticket['number']}: {ticket['subject']}\n\n{draft}")
        if POST_AS_NOTE:
            cooper(f"/tickets/{ticket['id']}/notes",
                   {"body": f"AI draft reply ({GEMINI_MODEL}, review before sending):\n\n{draft}", "is_public": False})
            print("\nPosted as an internal Note.")